# Part 1: Data Assessment

## Objective

Examine the provided arXiv research papers, identify the available file types and metadata,
assess the quality of the data, and determine what information is usable for the RAG pipeline.

In [1]:
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

cwd = Path.cwd()

if (cwd / "data").exists():
    data_dir = cwd / "data"
else:
    data_dir = cwd.parent / "data"

print("Data directory:", data_dir)

Data directory: /Users/eimansherzada/Desktop/BTT-KPMG/KPMG-1K-ai-research-intelligence-agent-for-business-insight-translation/data


In [2]:
files = sorted([
    file for file in data_dir.iterdir()
    if file.is_file() and file.name != ".gitkeep"
])

print("number of data files:", len(files))

for file in files:
    print(file.name)

number of data files: 5
2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pdf


In [3]:
file_info = []

for file in files:
    file_info.append({
        "filename": file.name,
        "file_type": file.suffix.lower(),
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2)
    })

file_df = pd.DataFrame(file_info)
file_df

,filename,file_type,size_mb
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,.pdf,0.59
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,.pdf,1.48
2,2608.20319v1_Inducing Task Models from Compute...,.pdf,4.67
3,2608.20320v1_An Agentic Approach for Active Da...,.pdf,2.58
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,.pdf,3.63


In [4]:
paper_info = []

for file in files:
    if file.suffix.lower() == ".pdf":
        try:
            reader = PdfReader(file)
            metadata = reader.metadata

            pages_with_text = 0
            total_characters = 0

            for page in reader.pages:
                text = page.extract_text()

                if text and text.strip():
                    pages_with_text += 1
                    total_characters += len(text)

            paper_info.append({
                "filename": file.name,
                "title": metadata.get("/Title") if metadata else None,
                "author": metadata.get("/Author") if metadata else None,
                "num_pages": len(reader.pages),
                "pages_with_text": pages_with_text,
                "text_characters": total_characters,
                "readable": total_characters > 0
            })

        except Exception as e:
            paper_info.append({
                "filename": file.name,
                "title": None,
                "author": None,
                "num_pages": None,
                "pages_with_text": 0,
                "text_characters": 0,
                "readable": False
            })

papers_df = pd.DataFrame(paper_info)
papers_df

,filename,title,author,num_pages,pages_with_text,text_characters,readable
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,28,28,89597,True
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,19,19,76707,True
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,23,23,85873,True
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,25,25,73593,True
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,9,9,48596,True


In [5]:
papers_df.isnull().sum()

filename           0
title              0
author             0
num_pages          0
pages_with_text    0
text_characters    0
readable           0
dtype: int64

In [6]:
papers_df[[
    "filename",
    "title",
    "author",
    "num_pages",
    "pages_with_text",
    "text_characters",
    "readable"
]]

,filename,title,author,num_pages,pages_with_text,text_characters,readable
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,28,28,89597,True
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,19,19,76707,True
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,23,23,85873,True
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,25,25,73593,True
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,9,9,48596,True


In [7]:
for file in files:
    reader = PdfReader(file)
    metadata = reader.metadata

    print("\nFILE:", file.name)

    if metadata:
        print("Metadata fields:", list(metadata.keys()))
    else:
        print("No metadata found")


FILE: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20319v1_Inducing Task Models from Computer-Use Traces.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20320v1_An Agentic Approach for Active Data Collection, Travel Behavior Modeling, and We.pdf
Metadata fields: ['/Author', '/Creator', '/DOI', '/License', '/PTEX.Fullbanner', '/Producer', '/Title', '/Trapped', '/arXivID']

FILE: 2608.20331v1_G-CARL Grounded Checklist-Aligned Reward Learning for Patient-Oriented Medical R.pd

In [8]:
reader = PdfReader(files[0])
sample_text = reader.pages[0].extract_text()

print(sample_text[:2000])

2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time
settings can improve quality and efficiency by routing queries to thespecialistwho can answer most
effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value
estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while ac-
curate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are
expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal
search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form
value-of-information expressions tha

In [9]:
for file in files:
    reader = PdfReader(file)
    text = reader.pages[0].extract_text()

    print("\nFILE:", file.name)
    print(text[:300])
    print("-" * 80)


FILE: 2608.20316v1_Pandora's AI Model Routing Box Efficient Allocation with Costly Value Estimation.pdf
2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architect
--------------------------------------------------------------------------------

FILE: 2608.20318v1_AI4AI-Bench Benchmarking LLM Agents in Algorithmic Design for Recursive Self-Imp.pdf
AI4AI-Bench: Benchmarking LLM Agents in
Algorithmic Design for Recursive Self-Improvement
Yizhe Chi †, Wenyi Li, Deyao Hong, Xiaoqiu Wang, Mingju Gao, Kaisen Yang, Bingxiang He, Y oujie Zheng, Calvin Xiao,
Qinhuai Na ‡
Navers Lab, Einsia.AI T singhua University
†Project Lead ‡Corresponding Author
Ab
--------------------------------------------------------------------------------

FILE: 2608.20319v1_Induci

In [10]:
print("duplicate filenames:", file_df["filename"].duplicated().sum())
print("duplicate titles:", papers_df["title"].duplicated().sum())

duplicate filenames: 0
duplicate titles: 0


In [11]:
metadata_rows = []

for file in files:
    reader = PdfReader(file)
    metadata = reader.metadata

    metadata_rows.append({
        "filename": file.name,
        "title": metadata.get("/Title"),
        "author": metadata.get("/Author"),
        "doi": metadata.get("/DOI"),
        "license": metadata.get("/License")
    })

metadata_df = pd.DataFrame(metadata_rows)
metadata_df

,filename,title,author,doi,license
0,2608.20316v1_Pandora's AI Model Routing Box Ef...,Pandora's AI Model Routing Box: Efficient Allo...,Adam Fisch; Shubhendu Trivedi; Fantine Huot; W...,https://doi.org/10.48550/arXiv.2608.20316,http://creativecommons.org/licenses/by/4.0/
1,2608.20318v1_AI4AI-Bench Benchmarking LLM Agen...,AI4AI-Bench: Benchmarking LLM Agents in Algori...,Yizhe Chi; Wenyi Li; Deyao Hong; Xiaoqiu Wang;...,https://doi.org/10.48550/arXiv.2608.20318,http://creativecommons.org/licenses/by/4.0/
2,2608.20319v1_Inducing Task Models from Compute...,Inducing Task Models from Computer-Use Traces,Yucheng Jiang; Zora Zhiruo Wang; Ruishi Chen; ...,https://doi.org/10.48550/arXiv.2608.20319,http://arxiv.org/licenses/nonexclusive-distrib...
3,2608.20320v1_An Agentic Approach for Active Da...,An Agentic Approach for Active Data Collection...,Narges Ahmadi; Yubo Jiao; Jônatas Augusto Manz...,https://doi.org/10.48550/arXiv.2608.20320,http://arxiv.org/licenses/nonexclusive-distrib...
4,2608.20331v1_G-CARL Grounded Checklist-Aligned...,G-CARL: Grounded Checklist-Aligned Reward Lear...,Shiao Xie; Siyu Chen; Jianwei Lv; Bo Yuan; Yuj...,https://doi.org/10.48550/arXiv.2608.20331,http://arxiv.org/licenses/nonexclusive-distrib...


## Data Quality Findings

- The dataset contains 5 unique arXiv research papers, all in PDF format.
- All 5 PDFs opened successfully and contained extractable text on every page.
- Titles and author information were available for all papers.
- DOI and license information were also available in the PDF metadata.
- No duplicate filenames or paper titles were found.
- No missing values were found in the fields collected during the assessment.
- The extracted text is readable overall, although some formatting artifacts are present, including missing spaces, merged words, and extra symbols around author names and affiliations.



### Usability for the RAG Pipeline

The following information should be useful for the RAG pipeline:
- Full paper text
- Paper titles
- Author information
- arXiv IDs from the filenames
- DOI and license information
- PDF page information that can be preserved later for citations and source tracking

The dataset looks usable for the RAG prototype. All 5 papers were readable, no duplicate filenames or titles were found, and the metadata needed for source tracking is available. Some formatting issues were found in the extracted text, including missing spaces, merged words, and extra symbols around author names and affiliations. These issues should be handled during the data cleaning and preprocessing stage before the text is split into chunks for retrieval.